# 02 - summarizing each phase with SQL

Goal: turn the 12-week dataset into one row per phase (baseline, risky lever, sustainable lever), using a SQL query, for the "historical cost of pushing" comparison chart.

Input: team_weekly.csv, uploaded below (the file notebook 01 produced)
Output: team_phase_summary.csv, 3 rows, downloaded at the end of this notebook

The transition week (week 6) is left out of this summary, it's a one-week pivot point, not one of the three real phases.

I'm using Python's built-in sqlite3 module so I don't need a real database server. The query itself would run the same way against an actual data warehouse.

This notebook is self-contained. It doesn't need notebook 01 to still be open, you just need the team_weekly.csv file it produced.

## Step 1 - upload team_weekly.csv

Run the cell below, then choose team_weekly.csv (the file notebook 01 made, you'll have downloaded it from that notebook already).

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import sqlite3
import pandas as pd

# use the actual uploaded filename, rather than assuming it's exactly
# "team_weekly.csv" -- if Colab had to rename it (e.g. "team_weekly (1).csv"
# because a file with that name already existed), this still works
uploaded_filename = list(uploaded.keys())[0]
wide = pd.read_csv(uploaded_filename)
wide.head()

## Step 2 - query: one row per phase, transition week excluded

WHERE phase != 'transition' drops the one-week pivot before grouping.
GROUP BY phase gives one row per phase.
AVG(...) gives the phase average for each metric.
The CASE phase WHEN ... bit just sorts the output in story order instead of alphabetical order.

In [ ]:
conn = sqlite3.connect(":memory:")
wide.to_sql("team_weekly", conn, index=False, if_exists="replace")

query = """
SELECT
    phase,
    CASE phase WHEN 'baseline' THEN 1 WHEN 'risky_lever' THEN 2 ELSE 3 END AS phase_order,
    COUNT(*)                                            AS weeks_in_phase,
    ROUND(AVG(tickets_received), 1)                     AS avg_tickets_received,
    ROUND(AVG(human_tickets_received), 1)                AS avg_human_tickets_received,
    ROUND(AVG(human_tickets_resolved_per_agent_day), 2)  AS avg_resolved_per_agent_day,
    ROUND(AVG(human_productivity_target_per_agent_day),2) AS avg_target_per_agent_day,
    ROUND(AVG(high_complexity_share_pct), 1)             AS avg_high_complexity_pct,
    ROUND(AVG(csat_score), 1)                            AS avg_csat,
    ROUND(AVG(wellbeing_pulse), 2)                       AS avg_wellbeing,
    ROUND(AVG(workload_distribution_pct), 1)             AS avg_workload_unfairness,
    ROUND(AVG(capacity_load_pct), 1)                     AS avg_capacity_load_pct,
    ROUND(AVG(actual_utilization_pct), 1)                AS avg_actual_utilization_pct,
    SUM(turnover_event)                                  AS turnover_events
FROM team_weekly
WHERE phase != 'transition'
GROUP BY phase
ORDER BY phase_order;
"""

phase_summary = pd.read_sql_query(query, conn)
conn.close()

phase_summary.to_csv("team_phase_summary.csv", index=False)
phase_summary

## Step 3 - reading the result

Tickets received stays roughly flat across phases, customer demand didn't change. Human-handled volume drops a lot once the AI takes over the routine tickets. Complexity share climbs and stays elevated in the sustainable phase too, that part is structural and doesn't reverse.

The part I want to be careful about explaining: CSAT, wellbeing, and workload fairness all partly recover in the sustainable-lever phase compared to risky-lever. But avg_resolved_per_agent_day stays far below avg_target_per_agent_day in both phases, and avg_capacity_load_pct doesn't drop in the sustainable phase, if anything it's slightly higher.

That's the finding I was expecting, not a flaw in the intervention. The sustainable lever was meant to make the human side of the work bearable again, not to undo the fact that the AI permanently changed how much easy work is left for people. A team can be working flat-out, high utilization, while also being in noticeably better shape on quality and wellbeing than it was during the risky-lever phase. Those are two different, related but separate things, and that's exactly what this project is trying to show.

## Step 4 - download the result

If you're running this in Google Colab, run the cell below to download team_phase_summary.csv to your computer.

In [ ]:
from google.colab import files

files.download("team_phase_summary.csv")